# Project Work NLP: Analisi della Distorsione e Omogeneizzazione della Scrittura da parte degli LLM

Questo notebook esplora l'impatto dei Modelli Linguistici di Grandi Dimensioni (LLM) sulla scrittura umana, implementando un'analisi basata su due recenti studi:

1. L'analisi di *"How LLMs Distort Our Written Language"* (2026), che dimostra come l'uso dell'IA come co-pilota di scrittura tenda a cancellare l'identità dell'autore, generando una "voce algoritmica" omogeneizzata.
2. L'indagine di *"Scientific production in the era of large language models"* (Science, 2025), che evidenzia come l'IA alteri la complessità linguistica spingendo verso un linguaggio iper-formale e artificiale.

Al fine di approfondire sperimentalmente i risultati emersi, questo project work è organizzato in tre esperimenti, ognuno dedicato all'analisi di uno specifico effetto degli LLM sulla produzione testuale.

* **Esperimento 1 - Entropia, Perplessità e Burstiness:** Analisi di come varia la prevedibilità statistica e la complessità strutturale del testo passando dalla bozza umana iniziale, alla revisione manuale, fino alla revisione artificiale (utilizzando il dataset *ArgRewrite*).
* **Esperimento 2 - Classificazione con BERT:** Addestramento di un modello Transformer (*DistilBERT*) per verificare se le firme stilistiche lasciate dall'IA sono sufficientemente marcate da essere distinte in un task di classificazione multiclasse.
* **Esperimento 3 - Collasso della Diversità Lessicale:** Replicazione della metodologia applicata alle recensioni *ICLR 2026* per misurare la perdita di varietà del vocabolario (tramite Type-Token Ratio) confrontando testi puramente umani e testi generati da IA su un dataset indipendente.

---

### Configurazione dell'Ambiente e Importazione delle Librerie
In questa cella viene configurato l'ambiente importando i pacchetti necessari per la manipolazione dei dati (`pandas`, `numpy`), la visualizzazione grafica (`matplotlib`, `seaborn`) e il calcolo delle metriche linguistiche avanzate (`transformers`, `nltk`, `textstat`).

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import nltk
import textstat

# download dei tokenizzatori NLTK per dividere il testo in frasi e parole
nltk.download('punkt', quiet=True)

# setup grafico per le curve evolutive temporali
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## Caricamento dei Dati e Allineamento Temporale

Prima di estrarre le metriche linguistiche, strutturiamo il dataset in modo da allineare ogni singolo saggio lungo il suo ciclo vitale. 

Attraverso l'integrazione del corpus **ArgRewrite V.2** (per i testi umani) e delle riscritture contenute in `dataframe_essays_llm_model.csv` (per i modelli generativi), costruiamo un unico DataFrame unificato (`df_timeline`). 

Ogni riga del DataFrame rappresenta un saggio tracciato in tre momenti storici:
1. **$t_0$ (Bozza Iniziale)**: Il testo grezzo scritto dallo studente prima di ricevere feedback (`Draft1`).
2. **$t_1$ (Revisione Umana)**: L'evoluzione naturale del saggio rivista dallo studente a seguito dei commenti (`Draft2`).
3. **$t_1^*$ (Riscrittura IA)**: La revisione parallela generata dall'LLM di riferimento a partire dalla bozza $t_0$ e dal feedback.

Questo allineamento ci permetterà di confrontare direttamente la traiettoria evolutiva dell'apprendimento umano contro la traiettoria di riscrittura sintetica dell'IA.

In [7]:
import os
import pandas as pd

# percorso
PATH_ANALYSIS = "./data/argrewrite_analysis"
csv_real_path = os.path.join(PATH_ANALYSIS, "dataframe_essays_llm_model.csv")

try:
    if os.path.exists(csv_real_path):
        # caricamento dataset
        df_raw = pd.read_csv(csv_real_path)
        
        # modelli disponibili nel CSV
        modelli_disponibili = df_raw["llm_model"].unique()
        print(f"[DATA] Modelli LLM presenti nel dataset: {list(modelli_disponibili)}")
        
        # seleziono il primo modello disponibile
        modello_target = modelli_disponibili[0]
        
        # se presente nel CSV, possiamo selezionare specificamente modello GPT-4o o Claude:
        df_filtered = df_raw[df_raw["llm_model"] == modello_target].copy()
        
        # costruzione del dataFrame timeline
        df_timeline = pd.DataFrame({
            "essay_id": df_filtered["essay_id"],
            "t0_bozza": df_filtered["draft1"],
            "t1_umano": df_filtered["draft2"],
            "t1_ia": df_filtered["ai_text"],
            "llm_model": df_filtered["llm_model"]
        }).dropna().reset_index(drop=True)
        
        print(f"\n[DATA] Allineamento completato con successo!")
        print(f"[DATA] Modello IA selezionato: '{modello_target}'")
        print(f"[DATA] Saggi caricati lungo la timeline: {len(df_timeline)}")
    else:
        raise FileNotFoundError(f"File non trovato in: {csv_real_path}")

except Exception as e:
    print(f"[AVVISO] Errore nel caricamento dei dati reali ({e}).")
    print("[AVVISO] Attivazione del dataset dummy per garantire la continuità dei test...")
    
    dummy_data = {
        "essay_id": [f"Saggio_{i}" for i in range(5)],
        "t0_bozza": [
            "This is a very simple and raw essay written by a student. It has errors.", 
            "Another essay with basic structure. Writing essays is hard for students.",
            "A third text containing some arguments but poor phrasing.",
            "Text number four. Arguments are missing clarity and depth.",
            "Final draft draft template. The punctuation is chaotic here!"
        ],
        "t1_umano": [
            "This is a simple, well-structured essay written by a student. It contains fewer errors.",
            "Another essay with an improved structure. Academic writing can be challenging.",
            "A third text containing clear arguments and refined phrasing.",
            "Text number four. The core arguments now feature clarity and depth.",
            "Final draft template. The overall punctuation has been corrected properly."
        ],
        "t1_ia": [
            "This represents a highly optimized and sophisticated academic essay featuring refined structure.",
            "An additional academic composition showcasing robust architecture and professional tone.",
            "A tertiary document containing advanced analytical insights and complex vocabulary.",
            "The fourth text exhibits absolute clarity, profound depth, and strategic organization.",
            "Concluding template matrix. Synthetic linguistic alignment has optimized the punctuation entirely."
        ]
    }
    df_timeline = pd.DataFrame(dummy_data)
    print(f"[DATA DUMMY] Caricato dataset di simulazione con {len(df_timeline)} righe.")

[DATA] Modelli LLM presenti nel dataset: ['CLAUDE-HAIKU-4-5-20251001', 'GPT-5-MINI', 'GEMINI-2.5-FLASH']

[DATA] Allineamento completato con successo!
[DATA] Modello IA selezionato: 'CLAUDE-HAIKU-4-5-20251001'
[DATA] Saggi caricati lungo la timeline: 860


## Esperimento 1 - Evoluzione Temporale delle Metriche Linguistiche

### Obiettivo
In questo primo esperimento vogliamo misurare quantitativamente come si trasforma l'impronta stilistica di un testo durante il suo processo di revisione. La letteratura scientifica suggerisce che l'intervento di un LLM tenda ad "appiattire" il testo, rendendolo statisticamente più prevedibile e strutturalmente monotono, aumentandone però al contempo la complessità del vocabolario.

Per dimostrarlo, analizzeremo l'evoluzione delle grandezze testuali confrontando tre fasi temporali distinte:
1. **$T_0$ (Bozza Iniziale):** Il saggio originale scritto dallo studente.
2. **$T_1$ Umano (Revisione Manuale):** Il saggio corretto dallo studente seguendo il feedback del docente.
3. **$T_1$ IA (Revisione LLM):** Il saggio originale fatto correggere a un modello IA utilizzando lo stesso feedback.

### Metriche Calcolate
Per tracciare questa evoluzione estrarremo quattro grandezze fondamentali:

* **Entropia (Cross-Entropy Loss):** Misura il grado di incertezza del modello nel prevedere la parola successiva. Testi molto "standardizzati" dall'IA presentano un'entropia minore.
* **Perplessità (Perplexity):** Direttamente derivata dall'entropia, indica quanto il testo sia "sorprendente" per un modello linguistico. Viene calcolata esponenzialmente:
  $$Perplexity = \exp(Entropy)$$
* **Burstiness (Esplosività Strutturale):** Valuta la varianza nella lunghezza delle frasi. Gli esseri umani alternano frasi brevi e lunghe (alta burstiness), mentre l'IA tende a produrre frasi di lunghezza uniforme. La calcoleremo tramite la deviazione standard del numero di parole per frase.
* **Leggibilità (Flesch Reading Ease):** Un indice per quantificare la difficoltà di lettura. L'obiettivo è verificare se la revisione dell'IA introduca una complessità sintattica artificiale rispetto alla revisione umana.